In [14]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [15]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_PROJECT'] = 'advance-rag'
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

## Part 1: Overview

In [16]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

In [17]:
website_urls = ("https://steemit.com/ocd/@albertocotua/brief-summary-of-what-happened-in-game-of-thrones", )
loader = WebBaseLoader(
    web_paths=website_urls,
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", 'post-title', 'post-header' )
    )
    )
)
docs = loader.load()

In [18]:
docs

[Document(metadata={'source': 'https://steemit.com/ocd/@albertocotua/brief-summary-of-what-happened-in-game-of-thrones'}, page_content='Brief summary of what happened in Game of Thrones.albertocotua (56)in #ocd • 7 years ago (edited)Pay attention that today I will tell you what Game of Thrones is about and a summary of EVERYTHING, what has happened before the last season to be released today.\nSource\nFor starters I tell you that the very successful series of HBO is based on the books written by George R.R. Martin, some books with several arguméntales lines that happen at the same time and that we will not take into account for this post. Because the series and the books have important differences. The books were originally going to be just one, then 3 and then 7, they stayed (for now) in the fifth, but let\'s focus on the series, since the material for this post comes from there, the books and extras of the BluRays, In addition, if one pays attention to the books you can notice the su

In [19]:
# Split - chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

In [20]:
#Embeddings

model_name = "BAAI/bge-large-en-v1.5"
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={
        "device": "cuda",  # Use RTX 4070
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32,
    },
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3877.96it/s]


In [21]:
## Vector store

vectorstore = FAISS.from_documents(documents=splits, 
                                    embedding=hf_embeddings)

In [29]:
#------------Retriever, prompt and generation--------------#

# Retriever
retriever = vectorstore.as_retriever()

# Prompt
prompt = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If the answer is not contained in the context, say "I don't know."
Keep the answer concise and limited to five sentences.
Context:
{context}
Question:
{question}
Answer:
""")

#LLM
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)

#Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = (
    {"context": retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

#Question
print(rag_chain.invoke('Why is Tywin Lannister critizied?'))

Tywin Lannister is not explicitly criticized in the context, but rather humiliated by the king. The king mocks Tywin and his pretensions when Tywin proposes a marriage between his daughter Cersei and the crown prince Rhaegar. Additionally, Tyrion's trial, which is judged by Tywin, is described as a farce, implying that Tywin's actions are unfair. However, the context does not provide a clear reason for criticism of Tywin. It seems that others, like the Bolton men, may dislike or hate Tyrion, not Tywin.


## Part 2: Indexing

In [24]:
#Load Docs
website_urls = ("https://steemit.com/ocd/@albertocotua/brief-summary-of-what-happened-in-game-of-thrones", )
loader = WebBaseLoader(
    web_paths=website_urls,
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", 'post-title', 'post-header' )
    )
    )
)
docs = loader.load()

# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap=50)

splits = text_splitter.split_documents(docs)

In [25]:
from langchain_huggingface import HuggingFaceEmbeddings
model_name = 'BAAI/bge-large-en-v1.5'
model_kwargs = {'device':'cuda'}
encode_kwargs = {
    "normalize_embeddings":True,
    'batch_size':32
}
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3975.76it/s]


In [26]:
question = "My cat is black in color"
document = "i have a blue dog"

query_result = hf_embeddings.embed_query(question)
document_result = hf_embeddings.embed_query(document)
print(len(query_result))

import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product/(norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("cosine_similarity: ", similarity)

1024
cosine_similarity:  0.6202689606020314


In [27]:
# Index
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents=splits, embedding=hf_embeddings)
retriever = vectorstore.as_retriever()

In [28]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020F59533350>, search_kwargs={})

## Part 3: Retrival

In [30]:
docs = retriever.invoke('Who is Tywin?')

for doc in docs:
    print(doc.page_content)
    print("#------------------------#")

That hand was a man named Tywin Lannister, who was constantly humiliated by the king because of the comments about Tywin driving the kingdom, but was offended and definitely distanced from the so-called Mad King when Tywin proposed to him that his daughter Cersei, He married the crown prince Rhaegar, but the mad king mocked Tywin and his pretensions.
#------------------------#
Tyrion Lannister, the dwarf brother of the queen decides to go with Jon Snow towards the wall to know the border between the civilized kingdom and the kingdom of the savages, and between him and Jon a strange friendship will be formed.
#------------------------#
managed to kill him and so he avenged himself for what he did to his fiancée. But after the battle Robert was very hurt so Eddard commanded the end of the rebellion and marched to the capital, all was lost to the Targaryen but suddenly Tywin Lannister appeared with his great army and the mad king ordered to be let in.
#------------------------#
And Cat tr